In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.window import Window
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

session = get_active_session()


In [ ]:
df_user_portfolios = session.table("BIGDATA_DB.STAGING.USER_PORTFOLIOS")

df_user_portfolios.limit(5).show()
print("Rows:", df_user_portfolios.count())
print("Columns:", df_user_portfolios.columns)

In [ ]:
df_user_token_count = (
    df_user_portfolios
    .group_by("USER_ID")
    .agg(
        F.count_distinct("TOKEN_ID").alias("TOKEN_COUNT"),
        F.sum("TX_COUNT").alias("TOTAL_TX_COUNT"),
    )
)

df_user_token_count.limit(20).show()
print("So user:", df_user_token_count.count())

## Histogram

In [ ]:
df_pandas = (
    df_user_token_count
    .order_by(F.desc("total_tx_count"))
    .to_pandas()
)

df_pandas.columns = [c.lower() for c in df_pandas.columns]
df_pandas = df_pandas.sort_values("total_tx_count", ascending=False).reset_index(drop=True)

df_pandas.head()

In [ ]:
num_users = df_user_portfolios.select("user_id").distinct().count()
num_tokens = df_user_portfolios.select("token_id").distinct().count()
num_interactions = df_user_portfolios.select("user_id", "token_id").distinct().count()

total_possible_interactions = num_users * num_tokens

sparsity = 1 - (num_interactions / total_possible_interactions)
density = num_interactions / total_possible_interactions

print("num_users:", num_users)
print("num_tokens:", num_tokens)
print("num_interactions:", num_interactions)
print("total_possible_interactions:", total_possible_interactions)
print("density:", density)
print("sparsity:", sparsity)

## Tạo bảng lọc token dựa trên số lượng người cùng hold 

In [ ]:
import pandas as pd
import snowflake.snowpark.functions as F

thresholds = [10, 20, 30, 40, 50]
summary_rows = []

df_token_holder_count = (
    df_user_portfolios
    .select("user_id", "token_id")
    .filter(
        F.col("user_id").is_not_null() &
        F.col("token_id").is_not_null()
    )
    .distinct()
    .group_by("token_id")
    .agg(F.count_distinct("user_id").alias("holder_count"))
)

for min_holders in thresholds:
    df_valid_tokens = (
        df_token_holder_count
        .filter(F.col("holder_count") >= min_holders)
        .select("token_id")
    )

    user_portfolios_filtered = (
        df_user_portfolios
        .join(df_valid_tokens, on="token_id", how="inner")
    )

    num_users = user_portfolios_filtered.select("user_id").distinct().count()
    num_tokens = user_portfolios_filtered.select("token_id").distinct().count()
    num_interactions = user_portfolios_filtered.select("user_id", "token_id").distinct().count()

    total_possible = num_users * num_tokens
    density = num_interactions / total_possible if total_possible > 0 else 0
    sparsity = 1 - density if total_possible > 0 else 1

    summary_rows.append({
        "min_holders": min_holders,
        "num_users": num_users,
        "num_tokens": num_tokens,
        "num_interactions": num_interactions,
        "density": density,
        "sparsity": sparsity,
    })

df_summary = pd.DataFrame(summary_rows)

df_summary.style.format({
    "num_users": "{:,}",
    "num_tokens": "{:,}",
    "num_interactions": "{:,}",
    "density": "{:.6f}",
    "sparsity": "{:.6f}",
})

## Lọc những token >= 30

In [ ]:
import snowflake.snowpark.functions as F

min_holders = 30

df_token_holder_count = (
    df_user_portfolios
    .select("user_id", "token_id")
    .filter(
        F.col("user_id").is_not_null() &
        F.col("token_id").is_not_null()
    )
    .distinct()
    .group_by("token_id")
    .agg(F.count_distinct("user_id").alias("holder_count"))
)

df_valid_tokens = (
    df_token_holder_count
    .filter(F.col("holder_count") >= min_holders)
    .select("token_id")
)

user_portfolios_filtered = (
    df_user_portfolios
    .join(
        df_valid_tokens,
        on="token_id",
        how="inner"
    )
)

print(user_portfolios_filtered.count())
user_portfolios_filtered.show(20)

In [ ]:
user_portfolios_filtered.write.mode("overwrite").save_as_table("BIGDATA_DB.STAGING.USER_PORTFOLIOS")

In [ ]:
from snowflake.snowpark import functions as F

df_user_token_count = (
    df_user_portfolios
    .group_by("user_id")
    .agg(F.count_distinct("token_id").alias("token_count"))
)

avg_token_per_user = (
    df_user_token_count
    .agg(F.avg("token_count").alias("avg_token_per_user"))
)

avg_token_per_user.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pdf_token_dist = df_token_count_distribution_pct.to_pandas()
pdf_token_dist.columns = [c.lower() for c in pdf_token_dist.columns]
pdf_token_dist["token_count"] = pd.to_numeric(pdf_token_dist["token_count"])

bins = [0, 1, 2, 3, 4, 5, 10, 20, 50, 100, np.inf]
labels = ["1", "2", "3", "4", "5", "6-10", "11-20", "21-50", "51-100", "100+"]

pdf_token_dist["bucket"] = pd.cut(
    pdf_token_dist["token_count"],
    bins=bins,
    labels=labels,
    right=True
)

pdf_bucket = (
    pdf_token_dist
    .groupby("bucket", observed=False, as_index=False)
    .agg(num_users=("num_users", "sum"), percentage=("percentage", "sum"))
)

pdf_bucket["bucket"] = pd.Categorical(pdf_bucket["bucket"], categories=labels, ordered=True)
pdf_bucket = pdf_bucket.sort_values("bucket")

colors = [
    "#FCE7D7", "#F8CFAF", "#F3B482", "#EA985B", "#D97B42",
    "#BF6037", "#9F4935", "#7D3734", "#5A2930", "#3A1D27"
]

fig, ax = plt.subplots(figsize=(11, 5), facecolor="#FAFAF7")
ax.set_facecolor("#FAFAF7")

bars = ax.bar(
    pdf_bucket["bucket"].astype(str),
    pdf_bucket["percentage"],
    color=colors,
    width=0.68,
    edgecolor="#FFFFFF",
    linewidth=1.2
)

ax.set_title(
    "User Distribution by Number of Tokens Held",
    fontsize=15,
    fontweight="bold",
    color="#222222",
    pad=14
)
ax.set_xlabel("Number of distinct tokens per user", fontsize=11, color="#444444")
ax.set_ylabel("Users (%)", fontsize=11, color="#444444")

ax.grid(axis="y", alpha=0.22, color="#777777")
ax.set_axisbelow(True)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color("#DDDDDD")
ax.spines["bottom"].set_color("#DDDDDD")

ax.tick_params(axis="x", colors="#444444")
ax.tick_params(axis="y", colors="#444444")

ax.set_ylim(0, pdf_bucket["percentage"].max() * 1.18)

for bar, pct in zip(bars, pdf_bucket["percentage"]):
    if pct >= 0.1:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{pct:.2f}%",
            ha="center",
            va="bottom",
            fontsize=9,
            color="#333333"
        )

plt.tight_layout()
plt.show()

display(pdf_bucket)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pdf_tx_dist = df_tx_count_distribution_pct.to_pandas()
pdf_tx_dist.columns = [c.lower() for c in pdf_tx_dist.columns]
pdf_tx_dist["total_tx_count"] = pd.to_numeric(pdf_tx_dist["total_tx_count"])

bins = [0, 1, 2, 5, 10, 20, 50, 100, 500, 1000, np.inf]
labels = ["1", "2", "3-5", "6-10", "11-20", "21-50", "51-100", "101-500", "501-1000", "1000+"]

pdf_tx_dist["bucket"] = pd.cut(
    pdf_tx_dist["total_tx_count"],
    bins=bins,
    labels=labels,
    right=True
)

pdf_tx_bucket = (
    pdf_tx_dist
    .groupby("bucket", observed=False, as_index=False)
    .agg(num_users=("num_users", "sum"), percentage=("percentage", "sum"))
)

pdf_tx_bucket["bucket"] = pd.Categorical(pdf_tx_bucket["bucket"], categories=labels, ordered=True)
pdf_tx_bucket = pdf_tx_bucket.sort_values("bucket")

colors = [
    "#D6F5F2", "#B8E8E3", "#8FD8D2", "#5FC3C0", "#35AEB8",
    "#2A93B5", "#2878A8", "#305F96", "#37497F", "#3E3568"
]

fig, ax = plt.subplots(figsize=(11, 5), facecolor="#FAFAF7")
ax.set_facecolor("#FAFAF7")

bars = ax.bar(
    pdf_tx_bucket["bucket"].astype(str),
    pdf_tx_bucket["percentage"],
    color=colors,
    width=0.68,
    edgecolor="#FFFFFF",
    linewidth=1.2
)

ax.set_title(
    "User Distribution by Total Transaction Count",
    fontsize=15,
    fontweight="bold",
    color="#222222",
    pad=14
)
ax.set_xlabel("Total transactions per user", fontsize=11, color="#444444")
ax.set_ylabel("Users (%)", fontsize=11, color="#444444")

ax.grid(axis="y", alpha=0.22, color="#777777")
ax.set_axisbelow(True)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["left"].set_color("#DDDDDD")
ax.spines["bottom"].set_color("#DDDDDD")

ax.tick_params(axis="x", colors="#444444")
ax.tick_params(axis="y", colors="#444444")

ax.set_ylim(0, pdf_tx_bucket["percentage"].max() * 1.18)

for bar, pct in zip(bars, pdf_tx_bucket["percentage"]):
    if pct >= 0.1:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{pct:.2f}%",
            ha="center",
            va="bottom",
            fontsize=9,
            color="#333333"
        )

plt.tight_layout()
plt.show()

display(pdf_tx_bucket)